# UGTM Plagiarism AI — Remote GPU Inference Worker

Notebook này thiết lập một dịch vụ suy luận GPU từ xa trên hạ tầng Google Colab (NVIDIA Tesla T4), đảm nhận việc tính toán hai mô hình học sâu:
1. **Dense Vector Embedding:** `BAAI/bge-m3` (1024 chiều)
2. **Cross-Encoder Reranking:** `BAAI/bge-reranker-v2-m3`

### Quy trình vận hành:
1. Chọn **Runtime** → **Change runtime type** → **Hardware accelerator: T4 GPU** → **Save**.
2. Thực thi toàn bộ các khối lệnh bên dưới (**Runtime** → **Run all**).
3. Sao chép địa chỉ `REMOTE_GPU_URL` xuất hiện ở cell cuối cùng và cập nhật vào file cấu hình `.env` của hệ thống máy chủ cục bộ.

In [ ]:
# 1. Cài đặt các gói thư viện phụ thuộc
!pip install -q FlagEmbedding fastapi uvicorn pycloudflared nest-asyncio requests

In [ ]:
# 2. Kiểm tra tài nguyên phần cứng GPU
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA device not detected. Please select Runtime -> Change runtime type -> T4 GPU.")

device_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print(f"Device: {device_name}")
print(f"VRAM: {total_vram_gb:.2f} GB")
print(f"CUDA Version: {torch.version.cuda}")
print(f"PyTorch Version: {torch.__version__}")

In [ ]:
# 3. Nạp mô hình học sâu lên bộ nhớ GPU (FP16)
import logging
from FlagEmbedding import BGEM3FlagModel, FlagReranker

logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("ugtm_gpu_worker")

logger.info("Loading embedding model: BAAI/bge-m3 onto CUDA...")
embedder = BGEM3FlagModel(
    "BAAI/bge-m3",
    use_fp16=True,
    devices=["cuda:0"],
)
logger.info("Embedding model BAAI/bge-m3 initialized successfully.")

logger.info("Loading reranker model: BAAI/bge-reranker-v2-m3 onto CUDA...")
reranker = FlagReranker(
    "BAAI/bge-reranker-v2-m3",
    use_fp16=True,
    devices=["cuda:0"],
)
logger.info("Reranker model BAAI/bge-reranker-v2-m3 initialized successfully.")

In [ ]:
# 4. Khởi tạo REST API và mở cổng kết nối từ xa (Cloudflare Tunnel)
import threading
import time
from typing import List
from fastapi import FastAPI
from pydantic import BaseModel, Field
import uvicorn
from pycloudflared import try_cloudflare

app = FastAPI(
    title="UGTM Remote GPU Inference Worker",
    version="1.0.0",
    description="Dedicated GPU inference endpoint for UGTM Plagiarism Engine",
)

class EncodeRequest(BaseModel):
    texts: List[str] = Field(..., description="List of input passage chunks to vectorize")
    batch_size: int = Field(default=32, ge=1, le=128)

class RerankRequest(BaseModel):
    pairs: List[List[str]] = Field(..., description="List of candidate pairs [query_text, source_text]")
    batch_size: int = Field(default=32, ge=1, le=128)

@app.get("/health")
def health_check():
    return {
        "status": "healthy",
        "device": torch.cuda.get_device_name(0),
        "cuda": True,
        "vram_allocated_mb": round(torch.cuda.memory_allocated(0) / (1024**2), 2),
    }

@app.post("/encode")
def encode_endpoint(payload: EncodeRequest):
    start_time = time.time()
    output = embedder.encode(
        payload.texts,
        batch_size=payload.batch_size,
        max_length=512,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )
    duration = time.time() - start_time
    throughput = len(payload.texts) / max(duration, 0.001)
    logger.info("Encoded %d passages in %.2fs (%.1f passages/s)", len(payload.texts), duration, throughput)
    return {"dense_vecs": output["dense_vecs"].tolist()}

@app.post("/rerank")
def rerank_endpoint(payload: RerankRequest):
    start_time = time.time()
    scores = reranker.compute_score(
        payload.pairs,
        batch_size=payload.batch_size,
        normalize=True,
        max_length=512,
    )
    duration = time.time() - start_time
    if isinstance(scores, (float, int)):
        normalized_scores = [float(scores)]
    else:
        normalized_scores = [float(s) for s in scores]
    logger.info("Reranked %d candidate pairs in %.2fs", len(payload.pairs), duration)
    return {"scores": normalized_scores}

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)

tunnel_instance = try_cloudflare(port=8000)
endpoint_url = tunnel_instance.tunnel

separator = "=" * 72
print("\n" + separator)
print("UGTM REMOTE GPU INFERENCE WORKER — ONLINE")
print(separator)
print(f"Target Hardware   : {device_name} ({total_vram_gb:.1f} GB VRAM)")
print(f"Public Endpoint   : {endpoint_url}")
print("\nSystem Configuration:")
print("Add or update the following entry in your local .env configuration:")
print(f"REMOTE_GPU_URL={endpoint_url}")
print(separator + "\n")

while True:
    time.sleep(60)